# Building Tower with topologic_fast

This notebook demonstrates creating a multi-story building tower using topologic_fast.
The example creates:
- A ground level with subdivided cells
- Columns supporting the structure
- A podium level
- A main building block
- A central core running through the building

We'll also create a connectivity graph showing how the cells relate to each other.

**Note**: This is adapted from the topologicpy Building_Tower example.

In [ ]:
import topologic_fast as tf
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## 1. Create Ground Level

Create a ground slab as a simple box below grade level.

In [ ]:
# Create ground slab
ground = tf.Cell.Box(0, 0, -1, 20, 20, 1)

print(f"Ground slab:")
print(f"  Volume: {ground.Volume():.2f} m^3")
print(f"  Area: {ground.Area():.2f} m^2")

## 2. Create Column Grid

Create a grid of columns to support the building. Columns are placed on a 4m grid.

In [ ]:
# Create columns on a 4m grid
columns = []
column_positions = []

for i in range(2, 20, 4):
    for j in range(2, 20, 4):
        # Create column as a box (0.8m x 0.8m x 3m)
        column = tf.Cell.Box(i - 0.4, j - 0.4, 0, 0.8, 0.8, 3)
        columns.append(column)
        column_positions.append((i, j))

print(f"Created {len(columns)} columns")
print(f"Column positions: {column_positions[:5]}...")

## 3. Create Podium Level

A podium spans most of the building footprint from level 3 to level 6.

In [ ]:
# Create podium (offset by 0.5m from edges)
podium = tf.Cell.Box(0.5, 0.5, 3, 19, 19, 3)

print(f"Podium:")
print(f"  Volume: {podium.Volume():.2f} m^3")
print(f"  Area: {podium.Area():.2f} m^2")

## 4. Create Main Building Block

The main tower rises from level 6 to level 21 (15m tall).

In [ ]:
# Create main building block
block = tf.Cell.Box(1, 1, 6, 18, 18, 15)

print(f"Main block:")
print(f"  Volume: {block.Volume():.2f} m^3")
print(f"  Area: {block.Area():.2f} m^2")

## 5. Create Core

A central core runs through the entire height of the building, housing elevators and stairs.

In [ ]:
# Create core (6m x 6m, full height of 24m)
core = tf.Cell.Box(7, 7, 0, 6, 6, 24)

print(f"Core:")
print(f"  Volume: {core.Volume():.2f} m^3")
print(f"  Area: {core.Area():.2f} m^2")

## 6. Visualization Helper Functions

In [ ]:
def cell_to_mesh_data(cell, color='lightblue', opacity=0.5, name='Cell'):
    """
    Convert a Cell to Plotly Mesh3d trace data.
    """
    traces = []
    faces = cell.Faces()
    
    all_x, all_y, all_z = [], [], []
    all_i, all_j, all_k = [], [], []
    vertex_idx = 0
    
    for face in faces:
        vertices = face.Vertices()
        if len(vertices) >= 3:
            coords = [v.Coordinates() for v in vertices]
            
            # Fan triangulation for convex polygons
            base_idx = vertex_idx
            for i, coord in enumerate(coords):
                all_x.append(coord[0])
                all_y.append(coord[1])
                all_z.append(coord[2])
                vertex_idx += 1
            
            # Create triangles
            for i in range(1, len(coords) - 1):
                all_i.append(base_idx)
                all_j.append(base_idx + i)
                all_k.append(base_idx + i + 1)
    
    return go.Mesh3d(
        x=all_x, y=all_y, z=all_z,
        i=all_i, j=all_j, k=all_k,
        color=color,
        opacity=opacity,
        name=name,
        flatshading=True
    )

def cell_to_edges(cell, color='black', width=1, name='Edges'):
    """
    Convert a Cell's edges to Plotly Scatter3d traces.
    """
    edges = cell.Edges()
    x_edges, y_edges, z_edges = [], [], []
    
    for edge in edges:
        verts = edge.Vertices()
        p1 = verts[0].Coordinates()
        p2 = verts[1].Coordinates()
        x_edges.extend([p1[0], p2[0], None])
        y_edges.extend([p1[1], p2[1], None])
        z_edges.extend([p1[2], p2[2], None])
    
    return go.Scatter3d(
        x=x_edges, y=y_edges, z=z_edges,
        mode='lines',
        line=dict(color=color, width=width),
        name=name,
        showlegend=False
    )

## 7. Visualize the Complete Building

In [ ]:
# Create figure
fig = go.Figure()

# Add ground
fig.add_trace(cell_to_mesh_data(ground, color='brown', opacity=0.7, name='Ground'))
fig.add_trace(cell_to_edges(ground, color='saddlebrown'))

# Add columns
for i, column in enumerate(columns):
    fig.add_trace(cell_to_mesh_data(column, color='gray', opacity=0.9, name=f'Column {i}'))
    fig.add_trace(cell_to_edges(column, color='darkgray'))

# Add podium
fig.add_trace(cell_to_mesh_data(podium, color='lightgreen', opacity=0.6, name='Podium'))
fig.add_trace(cell_to_edges(podium, color='green'))

# Add main block
fig.add_trace(cell_to_mesh_data(block, color='lightblue', opacity=0.5, name='Main Block'))
fig.add_trace(cell_to_edges(block, color='blue'))

# Add core
fig.add_trace(cell_to_mesh_data(core, color='orange', opacity=0.8, name='Core'))
fig.add_trace(cell_to_edges(core, color='darkorange'))

# Update layout
fig.update_layout(
    title='Building Tower - 3D View',
    scene=dict(
        aspectmode='data',
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.8))
    ),
    width=900,
    height=700,
    showlegend=True
)

fig.show()

## 8. Create CellComplex from Building Components

Combine all cells into a CellComplex for topological analysis.

In [ ]:
# Combine all cells
all_cells = [ground] + columns + [podium, block, core]

# Create CellComplex
building = tf.CellComplex.ByCells(all_cells)

print(f"Building CellComplex:")
print(f"  Number of cells: {building.NumCells()}")
print(f"  Total volume: {building.Volume():.2f} m^3")
print(f"  Total surface area: {building.Area():.2f} m^2")

## 9. Create Connectivity Graph

Create a dual graph where:
- Vertices represent cells (rooms/spaces)
- Edges represent shared walls/faces between cells

In [ ]:
# Create dual graph from CellComplex
graph = tf.Graph.ByTopology(building)

print(f"Connectivity Graph:")
print(f"  Vertices (cells): {graph.Order()}")
print(f"  Edges (connections): {graph.Size()}")
print(f"  Density: {graph.Density():.4f}")

## 10. Visualize Graph

Show the connectivity graph overlaid on the building.

In [ ]:
# Get graph vertices and edges
graph_vertices = graph.Vertices()
graph_edges = graph.Edges()

# Create figure
fig = go.Figure()

# Add building components (semi-transparent)
fig.add_trace(cell_to_mesh_data(ground, color='brown', opacity=0.2, name='Ground'))
for i, column in enumerate(columns):
    fig.add_trace(cell_to_mesh_data(column, color='gray', opacity=0.2, name=f'Column {i}'))
fig.add_trace(cell_to_mesh_data(podium, color='lightgreen', opacity=0.2, name='Podium'))
fig.add_trace(cell_to_mesh_data(block, color='lightblue', opacity=0.2, name='Main Block'))
fig.add_trace(cell_to_mesh_data(core, color='orange', opacity=0.2, name='Core'))

# Add graph edges
for edge in graph_edges:
    verts = edge.Vertices()
    p1 = verts[0].Coordinates()
    p2 = verts[1].Coordinates()
    fig.add_trace(go.Scatter3d(
        x=[p1[0], p2[0]],
        y=[p1[1], p2[1]],
        z=[p1[2], p2[2]],
        mode='lines',
        line=dict(color='red', width=4),
        showlegend=False
    ))

# Add graph vertices
vertex_coords = [v.Coordinates() for v in graph_vertices]
fig.add_trace(go.Scatter3d(
    x=[c[0] for c in vertex_coords],
    y=[c[1] for c in vertex_coords],
    z=[c[2] for c in vertex_coords],
    mode='markers',
    marker=dict(size=8, color='red', line=dict(color='darkred', width=1)),
    name='Graph Nodes'
))

# Update layout
fig.update_layout(
    title='Building with Connectivity Graph',
    scene=dict(
        aspectmode='data',
        xaxis_title='X (m)',
        yaxis_title='Y (m)',
        zaxis_title='Z (m)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=0.8))
    ),
    width=900,
    height=700
)

fig.show()

## 11. Building Statistics

In [ ]:
# Calculate building statistics
print("="*50)
print("BUILDING STATISTICS")
print("="*50)

# Component volumes
print("\nComponent Volumes:")
print(f"  Ground slab:    {ground.Volume():>10.2f} m^3")
column_volume = sum(c.Volume() for c in columns)
print(f"  Columns (x{len(columns)}):  {column_volume:>10.2f} m^3")
print(f"  Podium:         {podium.Volume():>10.2f} m^3")
print(f"  Main block:     {block.Volume():>10.2f} m^3")
print(f"  Core:           {core.Volume():>10.2f} m^3")
print(f"  ----------------------------------------")
total_volume = ground.Volume() + column_volume + podium.Volume() + block.Volume() + core.Volume()
print(f"  TOTAL:          {total_volume:>10.2f} m^3")

# Dimensions
print("\nBuilding Envelope:")
print(f"  Footprint:  20m x 20m = 400 m^2")
print(f"  Height:     24m (from grade to top)")

# Graph statistics  
print("\nConnectivity Graph:")
print(f"  Nodes (cells):    {graph.Order()}")
print(f"  Edges (shared):   {graph.Size()}")
print(f"  Graph density:    {graph.Density():.4f}")
print(f"  Is bipartite:     {graph.IsBipartite()}")

## 12. Floor Plan View

Create a 2D plan view of the building at different levels.

In [ ]:
# Create 2D plan view
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Level 0 (Ground)', 'Level 4 (Podium)', 'Level 10 (Tower)'),
    horizontal_spacing=0.1
)

def add_cell_plan(fig, cell, row, col, color, name):
    """Add 2D plan of a cell's bottom face."""
    faces = cell.Faces()
    for face in faces:
        vertices = face.Vertices()
        coords = [v.Coordinates() for v in vertices]
        z_coords = [c[2] for c in coords]
        # Find bottom face (all z coords equal to min)
        if all(abs(z - min(z_coords)) < 0.01 for z in z_coords):
            x = [c[0] for c in coords] + [coords[0][0]]
            y = [c[1] for c in coords] + [coords[0][1]]
            fig.add_trace(
                go.Scatter(x=x, y=y, fill='toself', fillcolor=color,
                          line=dict(color='black', width=1), name=name,
                          showlegend=False),
                row=row, col=col
            )
            break

# Level 0 - Ground and columns
add_cell_plan(fig, ground, 1, 1, 'brown', 'Ground')
for column in columns:
    add_cell_plan(fig, column, 1, 1, 'gray', 'Column')
add_cell_plan(fig, core, 1, 1, 'orange', 'Core')

# Level 4 - Podium
add_cell_plan(fig, podium, 1, 2, 'lightgreen', 'Podium')
add_cell_plan(fig, core, 1, 2, 'orange', 'Core')

# Level 10 - Tower
add_cell_plan(fig, block, 1, 3, 'lightblue', 'Block')
add_cell_plan(fig, core, 1, 3, 'orange', 'Core')

# Update layout
fig.update_layout(
    title='Building Floor Plans',
    height=400,
    width=1200
)

for i in range(1, 4):
    fig.update_xaxes(scaleanchor=f'y{i}' if i > 1 else 'y', scaleratio=1, 
                     title='X (m)', range=[-1, 22], row=1, col=i)
    fig.update_yaxes(title='Y (m)', range=[-1, 22], row=1, col=i)

fig.show()

## Summary

This notebook demonstrated:

1. **Creating Building Components** - Using `tf.Cell.Box()` to create various building elements
2. **Assembling a CellComplex** - Combining cells into a `CellComplex` for topological operations
3. **Graph Generation** - Using `tf.Graph.ByTopology()` to create a connectivity graph
4. **3D Visualization** - Using Plotly to visualize the building and graph
5. **Plan Views** - Creating 2D floor plans at different levels

### Notes on topologicpy vs topologic_fast differences:

- **CellComplex.Prism**: topologic_fast uses `tf.Cell.Box()` directly instead of `CellComplex.Prism()`
- **Topology.Translate**: Use `tf.Topology.TranslateCell()` for cells (type-specific transforms)
- **Topology.Difference**: Boolean operations work on Cells with `tf.Topology.Difference()`
- **Dictionary operations**: Not yet fully implemented for attaching metadata to topologies
- **Topology.Show**: Use plotly directly for visualization (more flexible)